# Gap 2b — TPU: λ_max During Training (ThermoRG)

## Objective
Measure how λ_max scales with width D at different training checkpoints.
Key question: does the GELU saturation effect (λ_max ∝ D^b, b≈0.26) emerge during training?

## TPU Optimizations

| Aspect | Strategy |
|--------|----------|
| Depth | 3 (vs 4) for faster training |
| Checkpoints | [0, 1, 10, 50] (drop 100) |
| Batch size | 512 (TPU-optimized) |
| Data | Synthetic random tensors (no I/O) |
| Seeds | 3 (vs 5) for speed |
| Training | SGD + momentum + xm.optimizer_step |

**Expected:** At init b≈0.5 (random weights); after training b may decrease toward 0.26.

## Step 1: TPU Setup

In [ ]:
# TPU setup
import os
import torch
import torch_xla
import torch_xla.core.xla_model as xm
import torch_xla.distributed.parallel_loader as pl

try:
    DEVICE = xm.xla_device()
    test_tensor = torch.zeros((2, 3), device=DEVICE)
    print('TPU device:', DEVICE)
    print('TPU runtime: OK')
except Exception as e:
    raise RuntimeError(
        f'TPU not accessible: {e}. '
        'Go to Notebook Settings → Accelerator → TPU → Save. '
        'Then restart (Kernel → Restart & clear output).'
    )

## Step 2: Clone + Install

In [ ]:
%cd /kaggle/working
!rm -rf ThermoRG-NN
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
gh_token = user_secrets.get_secret("gh_token")
repo_url = f"https://{gh_token}@github.com/xliu203/ThermoRG-NN.git"
!git clone {repo_url} --branch develop
%cd /kaggle/working/ThermoRG-NN
!git config --global user.email "xliu203@asu.edu"
!git config --global user.name "Leo Liu"
print("Cloned develop branch")

## Step 3: Imports + Config

In [ ]:
import sys, os, json, math, time, copy
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import stats

sys.path.insert(0, '/kaggle/working/ThermoRG-NN')

plt.rcParams.update({'font.size': 13, 'axes.labelsize': 14, 'axes.titlesize': 14,
                     'figure.figsize': (6, 4.5), 'figure.dpi': 130})

WORK_DIR = '/kaggle/working/ThermoRG-NN/experiments/phase_gap2'
os.makedirs(WORK_DIR, exist_ok=True)

# ── TPU-optimized config ──────────────────────────────────────────────────
WIDTHS = [128, 256, 512]
DEPTH = 3           # reduced from 4 for speed
CHECKPOINT_EPOCHS = [0, 1, 10, 50]  # drop 100
N_EPOCHS = 50
BATCH_SIZE = 512    # large batch for TPU efficiency
N_SEEDS = 3         # reduced from 5
LR = 0.01            # SGD learning rate
MOMENTUM = 0.9

N_ITER_PI = 20      # Power iteration steps

print(f"Config: WIDTHS={WIDTHS}, DEPTH={DEPTH}, EPOCHS={N_EPOCHS}")
print(f"       CHECKPOINT_EPOCHS={CHECKPOINT_EPOCHS}, N_SEEDS={N_SEEDS}")
print(f"       BATCH_SIZE={BATCH_SIZE}, LR={LR}, MOMENTUM={MOMENTUM}")

## Step 4: Model + Power Iteration

In [ ]:
class ThermoNet(nn.Module):
    """Conv2d + GELU network — same as Gap 2b but with configurable depth."""
    def __init__(self, width=128, depth=3, in_channels=3, num_classes=10,
                 kernel_size=3, padding=1):
        super().__init__()
        self.width = width
        self.depth = depth
        layers = []
        c = in_channels
        for i in range(depth):
            layers.append(nn.Conv2d(c, width, kernel_size=kernel_size, padding=padding))
            layers.append(nn.GELU())
            c = width
        layers += [
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(width, num_classes)
        ]
        self.net = nn.Sequential(*layers)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.Linear)):
                nn.init.normal_(m.weight, mean=0.0, std=1.0)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.net(x)


def power_iteration(W, n_iterations=20):
    """Estimate λ_max via power iteration. Works on CPU tensors."""
    W_flat = W.reshape(W.shape[0], -1)
    d = W_flat.shape[1]
    v = torch.randn(d)
    v = v / (v.norm() + 1e-10)
    for _ in range(n_iterations):
        Wv = torch.matmul(W_flat, v)
        v_new = torch.matmul(W_flat.T, Wv)
        v_norm = v_new.norm() + 1e-10
        v = v_new / v_norm
    return float(torch.matmul(W_flat, v).norm().item())


def compute_layer_stats(model):
    """Compute λ_max and D_eff for all Conv2d/Linear layers (CPU).
    
    Handles Edge of Stability regime where λ_max may approach zero.
    Clamp λ_max ≥ 1e-6 and cap D_eff ≤ 1e6 to prevent numerical overflow.
    """
    layer_lm, layer_de = [], []
    for _, module in model.named_modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            W_flat = module.weight.data.reshape(module.weight.shape[0], -1)
            lambda_max = power_iteration(module.weight.data, n_iterations=N_ITER_PI)
            # Clamp λ_max to prevent numerical explosion in EoS regime
            lambda_max = max(lambda_max, 1e-6)
            fro_sq = float((W_flat ** 2).sum().item())
            # D_eff = frobenius_norm^2 / lambda_max^2, capped to avoid overflow
            D_eff = min(fro_sq / (lambda_max ** 2), 1e6)
            layer_lm.append(lambda_max)
            layer_de.append(D_eff)
    return {
        'layer_lambda_max': layer_lm,
        'layer_D_eff': layer_de,
        'mean_lambda_max': float(np.mean(layer_lm)) if layer_lm else 0.0,
        'mean_D_eff': float(np.mean(layer_de)) if layer_de else 0.0,
    }


# Smoke test on CPU (before moving to TPU)
test_net = ThermoNet(width=128, depth=DEPTH)
x = torch.randn(2, 3, 32, 32)
out = test_net(x)
print(f"ThermoNet(128, {DEPTH}) output: {out.shape}")
stats_out = compute_layer_stats(test_net)
print(f"Mean λ_max: {stats_out['mean_lambda_max']:.4f}")
print(f"Mean D_eff: {stats_out['mean_D_eff']:.2f}")

## Step 5: Synthetic Data Loader (no CIFAR I/O)

In [ ]:
# ── Synthetic data: random tensors replace CIFAR-10 ────────────────────────
# Shape: (N, 3, 32, 32), labels: random integers 0-9
# This eliminates data loading I/O bottleneck on TPU

N_TRAIN = 10000
N_TEST = 2000

# Fixed seed for reproducibility of data generation
torch.manual_seed(0)
np.random.seed(0)

X_train = torch.randn(N_TRAIN, 3, 32, 32)
y_train = torch.randint(0, 10, (N_TRAIN,))
X_test  = torch.randn(N_TEST, 3, 32, 32)
y_test  = torch.randint(0, 10, (N_TEST,))

train_ds = TensorDataset(X_train, y_train)
test_ds  = TensorDataset(X_test, y_test)

# TPU uses mp_loader (no num_workers)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)

print(f"Synthetic train: {len(train_ds)} × {BATCH_SIZE} batches")
print(f"Synthetic test:  {len(test_ds)}")

## Step 6: Power Law Fitting Utility

In [ ]:
def fit_power_law(Ds, lambda_maxs):
    """Fit λ_max = a * D^b on log-log. Returns {a, b, R2, std_err}."""
    log_D  = np.log(np.array(Ds, dtype=float))
    log_lm = np.log(np.array(lambda_maxs, dtype=float))
    slope, intercept, r_value, p_value, std_err = stats.linregress(log_D, log_lm)
    a = math.exp(intercept)
    return {
        'a': a,
        'b': slope,
        'R2': r_value**2,
        'std_err': std_err,
        'intercept': intercept,
    }

def print_fit(name, result):
    print(f"  {name}: λ_max = {result['a']:.4f} * D^{result['b']:.4f}"
          f"  (R²={result['R2']:.4f}, σ={result['std_err']:.4f})")

## Step 7: TPU Training Loop with Checkpoints

**Key TPU pattern:**
```python
model = model.to(xm.xla_device())
xm.optimizer_step(optimizer)   # replaces optimizer.step()
```

We train all widths × seeds sequentially, checkpointing λ_max at each specified epoch.

In [ ]:
# ── TPU Training ───────────────────────────────────────────────────────────

criterion = nn.CrossEntropyLoss()

# Structure: results[seed][width][epoch] = {lambda_max, D_eff, accuracy}
all_results = {}

total_start = time.time()

for seed in range(N_SEEDS):
    print(f"\n{'='*65}")
    print(f"SEED {seed}")
    print(f"{'='*65}")
    seed_results = {}

    for w_idx, width in enumerate(WIDTHS):
        print(f"\n  [Seed {seed}] Training width={width}, depth={DEPTH}")

        torch.manual_seed(seed)
        np.random.seed(seed)

        # Build fresh model and move to TPU
        model = ThermoNet(width=width, depth=DEPTH, num_classes=10)
        model = model.to(DEVICE)
        optimizer = optim.SGD(model.parameters(), lr=LR, momentum=MOMENTUM)

        epoch_stats = {}
        
        # ── Epoch loop ────────────────────────────────────────────────────
        for epoch in range(N_EPOCHS + 1):
            # ── Checkpoint: measure λ_max on TPU ──────────────────────
            if epoch in CHECKPOINT_EPOCHS:
                # Move model to CPU for power iteration (avoids TPU overhead)
                model_cpu = copy.deepcopy(model).cpu()
                layer_stats = compute_layer_stats(model_cpu)
                del model_cpu

                # Accuracy on synthetic test set
                model.eval()
                correct = 0
                total = 0
                with torch.no_grad():
                    for Xb, yb in test_loader:
                        Xb_tpu = Xb.to(DEVICE)
                        yb_tpu = yb.to(DEVICE)
                        out = model(Xb_tpu)
                        correct += (out.argmax(1) == yb_tpu).sum().item()
                        total += yb.size(0)
                acc = correct / total

                epoch_stats[epoch] = {
                    'lambda_max': layer_stats['mean_lambda_max'],
                    'D_eff': layer_stats['mean_D_eff'],
                    'layer_lambda_max': layer_stats['layer_lambda_max'],
                    'layer_D_eff': layer_stats['layer_D_eff'],
                    'accuracy': acc,
                }
                print(f"    Epoch {epoch:3d}: λ_max={layer_stats['mean_lambda_max']:.4f}, "
                      f"D_eff={layer_stats['mean_D_eff']:.2f}, acc={acc:.4f}",
                      flush=True)

            # ── Train one epoch ──────────────────────────────────────────
            if epoch < N_EPOCHS:
                model.train()
                epoch_loss = 0
                for Xb, yb in train_loader:
                    Xb_tpu = Xb.to(DEVICE)
                    yb_tpu = yb.to(DEVICE)

                    optimizer.zero_grad()
                    out = model(Xb_tpu)
                    loss = criterion(out, yb_tpu)
                    loss.backward()
                    xm.optimizer_step(optimizer)   # TPU-specific step

                    epoch_loss += loss.item() * Xb.size(0)

                avg_loss = epoch_loss / len(train_ds)
                if (epoch + 1) % 10 == 0:
                    print(f"    Epoch {epoch+1:3d}/{N_EPOCHS}: loss={avg_loss:.4f}",
                          flush=True)

        seed_results[width] = epoch_stats
        print(f"  ✓ Width {width} done (seed {seed})", flush=True)

    all_results[seed] = seed_results

total_time = time.time() - total_start
print(f"\n{'='*65}")
print(f"All training complete in {total_time/60:.1f} min")
print(f"{'='*65}")

## Step 8: Aggregate Results — Scaling Exponent b vs Epoch

In [ ]:
# ── Aggregate across seeds: mean λ_max per width per epoch ───────────────

# epoch_results[epoch] = {width: {mean_lm, std_lm, mean_D_eff, ...}}
epoch_results = {}

for epoch in CHECKPOINT_EPOCHS:
    epoch_data = {}
    for width in WIDTHS:
        lms = [all_results[s][width][epoch]['lambda_max'] for s in range(N_SEEDS)]
        des = [all_results[s][width][epoch]['D_eff'] for s in range(N_SEEDS)]
        accs = [all_results[s][width][epoch]['accuracy'] for s in range(N_SEEDS)]
        epoch_data[width] = {
            'mean_lambda_max': np.mean(lms),
            'std_lambda_max': np.std(lms),
            'mean_D_eff': np.mean(des),
            'std_D_eff': np.std(des),
            'mean_accuracy': np.mean(accs),
        }
    epoch_results[epoch] = epoch_data

# ── Fit λ_max = a * D^b for each epoch ────────────────────────────────────
epoch_fits = {}
for epoch in CHECKPOINT_EPOCHS:
    Ds  = WIDTHS
    lms = [epoch_results[epoch][w]['mean_lambda_max'] for w in WIDTHS]
    fit = fit_power_law(Ds, lms)
    epoch_fits[epoch] = fit
    print_fit(f"Epoch {epoch:3d}", fit)

print("\nAll fits complete.")

## Step 9: Plot — Exponent b vs Epoch

In [ ]:
# ── Plot 1: λ_max vs D for each epoch ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

ax = axes[0]
colors = plt.cm.viridis(np.linspace(0, 0.9, len(CHECKPOINT_EPOCHS)))
for i, epoch in enumerate(CHECKPOINT_EPOCHS):
    Ds  = WIDTHS
    lms = [epoch_results[epoch][w]['mean_lambda_max'] for w in WIDTHS]
    errs = [epoch_results[epoch][w]['std_lambda_max'] for w in WIDTHS]
    ax.errorbar(Ds, lms, yerr=errs, marker='o', capsize=4,
                label=f'Epoch {epoch}', color=colors[i])

    # Fit line
    fit = epoch_fits[epoch]
    D_fit = np.linspace(min(Ds)*0.9, max(Ds)*1.1, 100)
    ax.plot(D_fit, fit['a'] * D_fit**fit['b'], '--', color=colors[i], alpha=0.6)

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Width D')
ax.set_ylabel('Mean λ_max')
ax.set_title('λ_max vs D at Training Checkpoints')
ax.legend()
ax.grid(True, alpha=0.3)

# ── Plot 2: Exponent b vs epoch ───────────────────────────────────────────
ax2 = axes[1]
epochs_plot = list(CHECKPOINT_EPOCHS)
b_values    = [epoch_fits[e]['b'] for e in epochs_plot]
b_errs      = [epoch_fits[e]['std_err'] for e in epochs_plot]

ax2.errorbar(epochs_plot, b_values, yerr=b_errs, marker='s', capsize=5,
             color='tab:blue', linewidth=2, markersize=8)

# Reference lines
ax2.axhline(0.5, color='gray', linestyle='--', alpha=0.7, label='b=0.5 (init, MP)')
ax2.axhline(0.26, color='tab:red', linestyle=':', alpha=0.7, label='b=0.26 (GELU sat.)')

ax2.set_xlabel('Training Epoch')
ax2.set_ylabel('Scaling exponent b')
ax2.set_title('λ_max Scaling Exponent b vs Training')
ax2.set_xticks(epochs_plot)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{WORK_DIR}/lambda_evolution.png', dpi=130, bbox_inches='tight')
plt.show()
print(f"Saved: {WORK_DIR}/lambda_evolution.png")

## Step 10: Save Structured Results

In [ ]:
# ── Save results ──────────────────────────────────────────────────────────

save_dict = {
    'config': {
        'widths': WIDTHS,
        'depth': DEPTH,
        'checkpoint_epochs': CHECKPOINT_EPOCHS,
        'n_epochs': N_EPOCHS,
        'batch_size': BATCH_SIZE,
        'n_seeds': N_SEEDS,
        'lr': LR,
        'momentum': MOMENTUM,
        'n_iter_pi': N_ITER_PI,
    },
    'epoch_fits': {str(e): v for e, v in epoch_fits.items()},
    'epoch_results': {str(e): 
        {str(w): v for w, v in ep.items()} 
        for e, ep in epoch_results.items()
    },
    'raw_results': {str(s): 
        {str(w): {str(ep): v for ep, v in wp.items()}
         for w, wp in sr.items()}
        for s, sr in all_results.items()
    },
    'total_time_minutes': total_time / 60,
}

results_path = f'{WORK_DIR}/training_time_results.json'
with open(results_path, 'w') as f:
    json.dump(save_dict, f, indent=2)

print(f"Saved: {results_path}")

# Print summary
print("\n── Scaling exponent b across epochs ──")
for epoch in CHECKPOINT_EPOCHS:
    fit = epoch_fits[epoch]
    print(f"  Epoch {epoch:3d}: b = {fit['b']:.4f} ± {fit['std_err']:.4f}  (R²={fit['R2']:.4f})")

## Step 11: Git Push Results

In [ ]:
%cd /kaggle/working/ThermoRG-NN
!git add experiments/phase_gap2/phase_gap2b_tpu.ipynb experiments/phase_gap2/training_time_results.json experiments/phase_gap2/lambda_evolution.png
!git status

In [ ]:
# Uncomment to push:
# !git commit -m "phase_gap2b_tpu: λ_max during training with TPU optimizations"
# !git push origin develop